In [1]:
"""
Ingests data/faq.csv into the 'faq' Chroma collection.
Run once (or whenever the CSV changes): python ingest_faq.py
"""
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
CHROMA_DIR = "chroma_store"
COLLECTION  = "faq"
CSV_PATH    = os.path.join("data", "faq.csv")
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

In [3]:
def load_faq_documents(csv_path: str) -> list[Document]:
    df = pd.read_csv(csv_path)
    docs = []
    for _, row in df.iterrows():
        content = f"Q: {row['question']}\nA: {row['answer']}"
        docs.append(Document(
            page_content=content,
            metadata={"source": "faq", "category": row["category"], "faq_id": str(row["id"])},
        ))
    return docs

In [4]:
print("Loading FAQ documents...")
docs = load_faq_documents(CSV_PATH)
print(f"  {len(docs)} FAQ entries loaded.")

print("Initialising embedding model...")
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

print(f"Embedding and storing in Chroma collection '{COLLECTION}'...")
vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        collection_name=COLLECTION,
        persist_directory=CHROMA_DIR,
    )
print(f"  Done. {vectorstore._collection.count()} vectors stored.")

Loading FAQ documents...
  25 FAQ entries loaded.
Initialising embedding model...


d:\GitClones_new\AgenticAI_Study\codeBasics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2589.96it/s]


Embedding and storing in Chroma collection 'faq'...
  Done. 25 vectors stored.
